# Chapter 4.4: ML Pipelines with LLM Features

Goal: Build a complete text-to-prediction pipeline using LLM-extracted features, compare against baselines, and measure extraction quality.

### Topics:
- Complete pipeline: text → LLM extraction → features → ML model
- Comparing models with and without LLM features
- Feature importance analysis
- Data leakage in LLM-augmented pipelines
- Measuring extraction accuracy with spot-checks
- Systematic error analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## Quick Recap

- **Feature pipeline**: The end-to-end process of transforming raw data into ML-ready features
- **Baseline model**: A simple model to compare against — shows whether added complexity helps
- **Feature importance**: How much each feature contributes to a model's predictions
- **Data leakage**: When information from the test set leaks into training, inflating performance
- **Spot-check**: Manually verifying a sample of LLM extractions against ground truth
- **Extraction accuracy**: The percentage of LLM-extracted labels that match human labels

## Data

We have a pre-built dataset of 100 product reviews with both simple (hand-computable) features and LLM-extracted features. The target variable is whether a review was voted "helpful" by other customers.

**Note on simulated LLM output:** The LLM-extracted features were pre-generated. In a real workflow, you'd call an API to extract these from review text.

In [ ]:
# Generate the dataset with a fixed random seed for reproducibility
np.random.seed(42)
n = 100

# Simple features (computable without an LLM)
review_length = np.random.randint(20, 300, n)
word_count = (review_length / 5.5).astype(int) + np.random.randint(-3, 4, n)
exclamation_count = np.random.choice([0, 0, 0, 1, 1, 2, 3], n)

# LLM-extracted features (would come from API in real workflow)
sentiment = np.random.choice(["positive", "negative", "mixed"], n, p=[0.45, 0.30, 0.25])
is_detailed = np.random.choice([0, 1], n, p=[0.4, 0.6])
mentions_price = np.random.choice([0, 1], n, p=[0.5, 0.5])
mentions_specific_features = np.random.choice([0, 1], n, p=[0.35, 0.65])
has_comparison = np.random.choice([0, 1], n, p=[0.7, 0.3])

# Target: helpful_votes (influenced by features)
helpful_score = (
    0.3 * (review_length > 100).astype(int) +
    0.2 * is_detailed +
    0.15 * mentions_specific_features +
    0.15 * has_comparison +
    0.1 * (exclamation_count > 0).astype(int) +
    0.1 * mentions_price +
    np.random.normal(0, 0.15, n)
)
helpful_votes = (helpful_score > 0.45).astype(int) * np.random.randint(1, 20, n)

# Build the DataFrame
df = pd.DataFrame({
    "review_length": review_length,
    "word_count": word_count,
    "exclamation_count": exclamation_count,
    "sentiment": sentiment,
    "is_detailed": is_detailed,
    "mentions_price": mentions_price,
    "mentions_specific_features": mentions_specific_features,
    "has_comparison": has_comparison,
    "helpful_votes": helpful_votes
})

df.head()

In [ ]:
# Spot-check data: 10 reviews where we have both LLM and human labels
spot_check = pd.DataFrame({
    "review_id": range(1, 11),
    "llm_sentiment": ["positive", "negative", "mixed", "negative", "positive", 
                       "positive", "mixed", "positive", "negative", "mixed"],
    "human_sentiment": ["positive", "negative", "mixed", "negative", "positive",
                         "mixed", "negative", "positive", "negative", "positive"],
    "llm_is_detailed": [1, 1, 0, 1, 1, 0, 1, 0, 1, 0],
    "human_is_detailed": [1, 1, 1, 1, 1, 0, 0, 0, 1, 0],
    "review_type": ["straightforward", "straightforward", "sarcastic", "straightforward", "straightforward",
                     "sarcastic", "sarcastic", "straightforward", "straightforward", "sarcastic"]
})

spot_check

## Practice

### 1. By hand — Explore the dataset

Answer these questions about the data:
1. What's the distribution of sentiment labels?
2. What's the mean `helpful_votes` for each sentiment category?
3. What percentage of reviews have zero helpful votes?

In [ ]:
# Sentiment distribution
...

In [ ]:
# Mean helpful_votes by sentiment
...

In [ ]:
# Percentage of reviews with zero helpful votes
...

### 2. By hand — Create binary target and check balance

Create a new column `is_helpful` that is 1 if `helpful_votes > 0` and 0 otherwise. Check the class balance — is it roughly even or heavily skewed?

In [ ]:
# Create binary target
df["is_helpful"] = ...

# Check class balance
...

**Your observation:** Is the class balance acceptable for training? What problems might arise if it were heavily skewed?

(Write your answer here)

### 3. Use AI — Build two Random Forest models

Use your AI assistant to build two models:

**Baseline model:** Uses only simple features (`review_length`, `word_count`, `exclamation_count`)

**Full model:** Uses simple features + LLM-extracted features (`is_detailed`, `mentions_price`, `mentions_specific_features`, `has_comparison`, and one-hot encoded `sentiment`)

For both models:
- Use `train_test_split` with `test_size=0.2, random_state=42`
- Use `RandomForestClassifier(n_estimators=100, random_state=42)`
- Print accuracy and classification report

In [ ]:
# Prepare features — one-hot encode sentiment
df_encoded = pd.get_dummies(df, columns=["sentiment"], prefix="sentiment")

# Define feature sets
simple_features = ["review_length", "word_count", "exclamation_count"]
llm_features = ["is_detailed", "mentions_price", "mentions_specific_features", 
                "has_comparison", "sentiment_mixed", "sentiment_negative", "sentiment_positive"]
all_features = simple_features + llm_features

target = "is_helpful"

In [ ]:
# Use AI to build and compare the two models
# Baseline model (simple features only)
...

print("=== BASELINE MODEL (Simple Features Only) ===")
...

In [ ]:
# Full model (simple + LLM features)
...

print("=== FULL MODEL (Simple + LLM Features) ===")
...

### 4. Interpretation — Did LLM features help?

Compare the two models and look at feature importances for the full model.

In [ ]:
# Get feature importances from the full model
# (This assumes you named your full model something like full_model)
# Adjust the variable name to match what you used above
...

# Display as a sorted DataFrame
...

**Your analysis:**

1. Did adding LLM features improve accuracy? By how much?

(Write your answer here)

2. Which features are most important according to the Random Forest? Are any LLM features in the top 3?

(Write your answer here)

3. If the LLM features didn't help much, what might that tell us about this particular dataset?

(Write your answer here)

### 5. By hand — Data leakage scenario

Consider this scenario: You develop your LLM extraction prompt by looking at all 100 reviews and manually tuning the prompt until the extracted sentiments look correct. Then you split the data into train/test and build your model.

Is there a data leakage problem here? Why or why not?

**Your analysis:**

1. Is this data leakage? Explain why or why not.

(Write your answer here)

2. How would you prevent this in practice?

(Write your answer here)

3. Is this the same kind of leakage as fitting a scaler on the test set? How is it different?

(Write your answer here)

### 6. By hand — Calculate extraction accuracy

Using the `spot_check` DataFrame, calculate:
1. Overall sentiment accuracy (LLM vs human)
2. Overall `is_detailed` accuracy
3. Look at which reviews the LLM got wrong — do you see a pattern?

In [ ]:
# Calculate sentiment accuracy
sentiment_correct = ...
sentiment_accuracy = ...
print(f"Sentiment accuracy: {sentiment_accuracy:.0%}")

# Calculate is_detailed accuracy
detailed_correct = ...
detailed_accuracy = ...
print(f"Is_detailed accuracy: {detailed_accuracy:.0%}")

In [ ]:
# Show only the rows where the LLM got sentiment wrong
wrong_sentiment = spot_check[spot_check["llm_sentiment"] != spot_check["human_sentiment"]]
wrong_sentiment

**Your observation:** What type of reviews does the LLM struggle with? Look at the `review_type` column.

(Write your answer here)

### 7. Use AI — Extraction accuracy by category

Use your AI assistant to write a function `extraction_accuracy_by_category(spot_check_df, llm_col, human_col, group_col)` that:
1. Groups the spot-check data by a category column
2. Calculates accuracy for each group
3. Returns a DataFrame with category, accuracy, and count

Use it to find accuracy broken down by `review_type`.

In [ ]:
# Use AI to implement this function
def extraction_accuracy_by_category(spot_check_df, llm_col, human_col, group_col):
    ...

# Calculate accuracy by review_type for sentiment
accuracy_by_type = extraction_accuracy_by_category(
    spot_check, "llm_sentiment", "human_sentiment", "review_type"
)
accuracy_by_type

**Your observation:** Which review type has the worst extraction accuracy? Why does this make sense?

(Write your answer here)

### 8. Interpretation — Handling LLM errors in production

Your LLM consistently misclassifies sarcastic reviews as positive. You're using these extractions as features for a model that recommends reviews to shoppers.

**Your plan:** What concrete actions would you take? Consider:

1. Changes to the extraction prompt

(Write your answer here)

2. Changes to the ML pipeline

(Write your answer here)

3. Changes to validation/monitoring

(Write your answer here)

## Discussion

If your LLM extraction has 85% accuracy, but adding those features improves your ML model from 70% to 78% accuracy, are the noisy LLM features worth using? What factors would influence your decision?

(Discuss with a neighbor)